# External test of the physics-informed forecaster: train on MU, test on LUMIERE — notebook v1

**Before running:** the protocol `docs/preregistration_external_forecaster.md` and
`scripts/external_forecaster.py` must be committed **and pushed** to GitHub, because
this notebook clones the repository.

What it does:
1. Downloads LUMIERE masks and volumes into Drive (first run only, ~40 min, range
   requests on figshare, no 32 GB download).
2. Writes deduplicated, treatment-free MU indices and the LUMIERE external index.
3. Trains the forecaster on MU (3 seeds, 2 mm, 25 epochs), selecting the epoch on
   MU model selection. LUMIERE is never used for training or selection.
4. Evaluates the selected models unchanged on MU final test and LUMIERE.

Runtime: GPU (T4 or better). Results are written to
`STS_Project_Data/outputs/external_forecaster/`.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!rm -rf /content/BINNs
!git clone https://github.com/tanushappapogu-max/BINNs.git /content/BINNs
%cd /content/BINNs
!git log -1 --oneline
!pip install -e '.[ml,imaging,datasets,analysis]' -q

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# LUMIERE masks and per-session volumes (skipped if already on Drive).
import os

DRIVE = "/content/drive/MyDrive/STS_Project_Data"
volumes = f"{DRIVE}/outputs/surveillance/lumiere_scan_volumes.csv"
masks = f"{DRIVE}/data/raw/lumiere_masks"
if not os.path.exists(volumes) or not os.path.isdir(masks):
    os.makedirs(os.path.dirname(volumes), exist_ok=True)
    !python scripts/prepare_lumiere_volumes.py --output {volumes} --masks-dir {masks}
else:
    print("LUMIERE already on Drive")

In [ ]:
!python scripts/external_forecaster.py prepare --data-root {DRIVE} --lumiere-volumes {volumes}

In [ ]:
# Training: 3 seeds x 25 epochs. Use --downsample 3 if a T4 is too slow.
OUT = f"{DRIVE}/outputs/external_forecaster"
!python scripts/external_forecaster.py train --data-root {DRIVE} --output {OUT} \
    --downsample 2 --epochs 25 --batch-size 2 --device cuda --seeds 42 43 44

In [ ]:
!python scripts/external_forecaster.py evaluate --data-root {DRIVE} --output {OUT} \
    --downsample 2 --device cuda --seeds 42 43 44
!python scripts/judge_external_forecaster.py --output {OUT}

In [ ]:
import json

with open(f"{OUT}/external_verdicts.json") as handle:
    print(json.dumps(json.load(handle), indent=2))